# Instalando o Spark

In [1]:
!pip install pyspark #==3.3.1

In [2]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip ngrok-stable-linux-amd64.zip

--2026-02-25 22:55:57--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 75.2.60.68, 35.71.179.82, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|75.2.60.68|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13921656 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip.2’

ngrok-stable-linux- 100%[===================>]  13.28M  17.3MB/s    in 0.8s    

2026-02-25 22:55:58 (17.3 MB/s) - ‘ngrok-stable-linux-amd64.zip.2’ saved [13921656/13921656]

Archive:  ngrok-stable-linux-amd64.zip
replace ngrok? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


# Iniciar Sessão Spark

In [3]:
from pyspark.sql import SparkSession

# Stop any existing SparkSession to avoid conflicts
if 'spark' in locals() and spark.sparkContext._jsc is not None:
    spark.stop()

spark = (
    SparkSession.builder                  # Método da classe que constrói a sessão spark
      .appName("Meu Primeiro App Spark")  # Nome do App Spark
      .config('spark.ui.port', '4050')    # Configure UI port directly in SparkSession builder
      .getOrCreate())                     # Verifica se há uma sessão ativa, e se não há, cria uma nova sessão


In [4]:
!curl -s http://localhost:4040/api/tunnels

In [5]:
!pip install pyspark

In [6]:
import time

# Give ngrok some more time to establish the tunnel if it hasn't already
time.sleep(5)

# Retrieve and print the public URL for the Spark UI
ngrok_url = !curl -s http://localhost:4040/api/tunnels | grep -oP '(?<="public_url":")[^"]*'

if ngrok_url:
    print(f"Spark UI is available at: {ngrok_url[0]}")
else:
    print("Could not retrieve ngrok public URL. Please ensure ngrok is running.")

Could not retrieve ngrok public URL. Please ensure ngrok is running.


In [7]:
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip

In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4040')
      .appName("SparkUI Introdução")
      .getOrCreate()
)

In [9]:
!./ngrok authtoken
get_ipython().system_raw('./ngrok http 4040 &')
!sleep 10
!curl -s http://localhost:4040/api/tunnels | grep -Po 'public_url":"(?=https)\K[^"]*'

NAME:
   authtoken - save authtoken to configuration file

USAGE:
   ngrok authtoken [command options] [arguments...]

DESCRIPTION:
   The authtoken command modifies your configuration file to include
   the specified authtoken. By default, this configuration file is located
   at $HOME/.ngrok2/ngrok.yml

   The ngrok.com service requires that you sign up for an account to use
   many advanced service features. In order to associate your client with
   an account, it must pass a secret token to the ngrok.com service when it
   starts up. Instead of passing this authtoken on every invocation, you may
   use this command to save it into your configuration file so that your
   client always authenticates you properly.

EXAMPLE:
    ngrok authtoken BDZIXnhJt2HNWLXyQ5PM_qCaBq0W2sNFcCa0rfTZd

OPTIONS:
   --config 		save in this config file, default: ~/.ngrok2/ngrok.yml
   --log "false"	path to log file, 'stdout', 'stderr' or 'false'
   --log-format "term"	log record format: 'term', 'logfmt',

In [10]:
# {
#     "id_transacao": 1000,
#     "valor": "58931.97",
#     "remetente": {"nome": "Jonathan Gonsalves", "banco": "BTG", "tipo": "PF"},
#     "destinatario": {"nome": "Emanuella Moura", "banco": "Itau", "tipo": "PJ"},
#     "transaction_date": "2021-06-02",
#     "chave_pix": "aleatoria",
#     "fraude": "1"
# }

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, TimestampType

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType()),
])


schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('transaction_date', TimestampType()),
    StructField('chave_pix', StringType()),
    StructField('fraude', IntegerType())
])


caminho_json = '/content/pix_transactions.json'

df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
)

In [11]:
df.show()

+------------+--------+--------------------+--------------------+-------------------+---------+------+
|id_transacao|   valor|           remetente|        destinatario|   transaction_date|chave_pix|fraude|
+------------+--------+--------------------+--------------------+-------------------+---------+------+
|        1000|    7.05|{Jonathan Gonsalv...|{Gabriel Cunha, I...|2022-03-19 00:00:00|      cpf|     0|
|        1001|   37.28|{Jonathan Gonsalv...|{Diego Souza, XP,...|2021-01-26 00:00:00|aleatoria|     0|
|        1002|  282.73|{Jonathan Gonsalv...|{Nicole Nunes, BT...|2022-05-31 00:00:00|aleatoria|     0|
|        1003| 8447.92|{Jonathan Gonsalv...|{Maria Fernanda C...|2022-07-04 00:00:00|aleatoria|     0|
|        1004|   58.51|{Jonathan Gonsalv...|{Isabel Silva, C6...|2021-09-11 00:00:00|aleatoria|     0|
|        1005| 6655.12|{Jonathan Gonsalv...|{Anthony Carvalho...|2022-02-11 00:00:00|  celular|     0|
|        1006| 9912.25|{Jonathan Gonsalv...|{Eloah Monteiro, ...|2022-05-

In [12]:
df.select('destinatario.nome').show()

+--------------------+
|                nome|
+--------------------+
|       Gabriel Cunha|
|         Diego Souza|
|        Nicole Nunes|
|Maria Fernanda Ca...|
|        Isabel Silva|
|    Anthony Carvalho|
|      Eloah Monteiro|
|        Sophie Rocha|
|      Pietro Ribeiro|
|      Eloah Teixeira|
|     Emanuella Sales|
|    Valentina Campos|
|       Stella Araujo|
|     Benicio Costela|
|      Joao Fernandes|
|   Gabriela da Rocha|
|      Larissa Aragao|
|           Theo Dias|
|        Danilo Jesus|
|       Bruno Correia|
+--------------------+
only showing top 20 rows


In [13]:
df.write.mode('overwrite').partitionBy('chave_pix').parquet('outpute/pix')

# SparkSQL

In [14]:
!pip install pyspark

In [15]:
import time

# Give ngrok some more time to establish the tunnel if it hasn't already
time.sleep(5)

# Retrieve and print the public URL for the Spark UI
ngrok_url = !curl -s http://localhost:4040/api/tunnels | grep -oP '(?<="public_url":")[^"]*'

if ngrok_url:
    print(f"Spark UI is available at: {ngrok_url[0]}")
else:
    print("Could not retrieve ngrok public URL. Please ensure ngrok is running.")

Could not retrieve ngrok public URL. Please ensure ngrok is running.


In [16]:
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip

In [17]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4040')
      .appName("SparkUI Introdução")
      .getOrCreate()
)

In [18]:
# 1. Instala a ferramenta de conexão
!pip install pyngrok

from pyngrok import ngrok

# --- SEGURANÇA (COLE SEU TOKEN ABAIXO) ---
# Sem o token, o túnel vai cair rápido.
SEU_TOKEN = ""
ngrok.set_auth_token(SEU_TOKEN)

# 2. Mata processos velhos para limpar a área
ngrok.kill()

# 3. Descobre em qual porta o Spark está (Geralmente 4040, mas pode mudar)
try:
    # Pega a porta direto do SparkContext
    ui_url = spark.sparkContext.uiWebUrl
    port = ui_url.split(':')[-1]
    print(f"O Spark está rodando na porta interna: {port}")

    # 4. Abre o túnel para essa porta específica
    public_url = ngrok.connect(port).public_url
    print(f"\n🎉 CLIQUE AQUI PARA ABRIR O SPARK UI: {public_url}")

except Exception as e:
    print("Erro: O Spark não parece estar ativo. Rode 'spark = SparkSession.builder...' primeiro.")

O Spark está rodando na porta interna: 4050

🎉 CLIQUE AQUI PARA ABRIR O SPARK UI: https://cleopatra-arthritic-regerminatively.ngrok-free.dev


In [19]:
print("Checking ngrok tunnels API...")
!curl -s http://localhost:4040/api/tunnels

Checking ngrok tunnels API...
{"tunnels":[{"name":"http-4050-3c2dab77-b37f-4980-805c-7123693ce453","ID":"0a9f56b29d08e08bdc7c8bbcef6035b9","uri":"/api/tunnels/http-4050-3c2dab77-b37f-4980-805c-7123693ce453","public_url":"https://cleopatra-arthritic-regerminatively.ngrok-free.dev","proto":"https","config":{"addr":"http://localhost:4050","inspect":true},"metrics":{"conns":{"count":0,"gauge":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0},"http":{"count":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0}}}],"uri":"/api/tunnels"}


In [20]:
# {
#     "id_transacao": 1000,
#     "valor": "58931.97",
#     "remetente": {"nome": "Jonathan Gonsalves", "banco": "BTG", "tipo": "PF"},
#     "destinatario": {"nome": "Emanuella Moura", "banco": "Itau", "tipo": "PJ"},
#     "transaction_date": "2021-06-02",
#     "chave_pix": "aleatoria",
#     "fraude": "1"
# }

from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, TimestampType

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType()),
])


schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('transaction_date', TimestampType()),
    StructField('chave_pix', StringType()),
    StructField('fraude', IntegerType())
])


caminho_json = '/content/pix_transactions.json'

spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
).createOrReplaceTempView('transacoes_pix')


df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd"
)

In [21]:
spark.sql("select * from transacoes_pix limit 10").show()

+------------+-------+--------------------+--------------------+-------------------+---------+------+
|id_transacao|  valor|           remetente|        destinatario|   transaction_date|chave_pix|fraude|
+------------+-------+--------------------+--------------------+-------------------+---------+------+
|        1000|   7.05|{Jonathan Gonsalv...|{Gabriel Cunha, I...|2022-03-19 00:00:00|      cpf|     0|
|        1001|  37.28|{Jonathan Gonsalv...|{Diego Souza, XP,...|2021-01-26 00:00:00|aleatoria|     0|
|        1002| 282.73|{Jonathan Gonsalv...|{Nicole Nunes, BT...|2022-05-31 00:00:00|aleatoria|     0|
|        1003|8447.92|{Jonathan Gonsalv...|{Maria Fernanda C...|2022-07-04 00:00:00|aleatoria|     0|
|        1004|  58.51|{Jonathan Gonsalv...|{Isabel Silva, C6...|2021-09-11 00:00:00|aleatoria|     0|
|        1005|6655.12|{Jonathan Gonsalv...|{Anthony Carvalho...|2022-02-11 00:00:00|  celular|     0|
|        1006|9912.25|{Jonathan Gonsalv...|{Eloah Monteiro, ...|2022-05-10 00:00:0

In [22]:
group_sql = spark.sql('select chave_pix, count(*) from transacoes_pix group by chave_pix')

In [23]:
group_df = df.groupBy('chave_pix').count()

In [24]:
group_sql.explain()

group_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[chave_pix#56], functions=[count(1)])
   +- Exchange hashpartitioning(chave_pix#56, 200), ENSURE_REQUIREMENTS, [plan_id=74]
      +- HashAggregate(keys=[chave_pix#56], functions=[partial_count(1)])
         +- FileScan json [chave_pix#56] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/pix_transactions.json], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<chave_pix:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[chave_pix#63], functions=[count(1)])
   +- Exchange hashpartitioning(chave_pix#63, 200), ENSURE_REQUIREMENTS, [plan_id=87]
      +- HashAggregate(keys=[chave_pix#63], functions=[partial_count(1)])
         +- FileScan json [chave_pix#63] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/pix_transactions.json], PartitionFilters: [], PushedFilters: [], R

In [25]:
group_sql.show()

+---------+--------+
|chave_pix|count(1)|
+---------+--------+
|aleatoria|   25045|
|  celular|   24841|
|    email|   24935|
|      cpf|   25179|
+---------+--------+



In [26]:
group_df.show()

+---------+-----+
|chave_pix|count|
+---------+-----+
|aleatoria|25045|
|  celular|24841|
|    email|24935|
|      cpf|25179|
+---------+-----+



In [27]:
spark.sql("""
    select
        chave_pix,
        round(avg(valor), 3)
    from transacoes_pix
    group by 1
    order by 2 desc
""").show()

+---------+--------------------+
|chave_pix|round(avg(valor), 3)|
+---------+--------------------+
|aleatoria|           12217.234|
|  celular|            12152.68|
|      cpf|           11946.072|
|    email|           11868.017|
+---------+--------------------+



In [28]:
spark.sql("""
    select
        chave_pix,
        count(*) as count_maior_100
    from transacoes_pix
    where valor > 10000
    group by 1
    order by 1 desc
""").show()

+---------+---------------+
|chave_pix|count_maior_100|
+---------+---------------+
|    email|           4830|
|      cpf|           4950|
|  celular|           4922|
|aleatoria|           5032|
+---------+---------------+



In [29]:
spark.sql("""
    with cte_base_window as (
        select
            destinatario.banco,
            valor,
            row_number() over (partition by destinatario.banco order by valor desc) as row_number
        from transacoes_pix
    )
    select
        banco,
        valor
    from cte_base_window
    where row_number in (1, 2)
""").show()

+--------+--------+
|   banco|   valor|
+--------+--------+
|     BTG|99946.78|
|     BTG| 99913.9|
|Bradesco|99910.87|
|Bradesco|99887.88|
|      C6|99980.03|
|      C6|99964.99|
|   Caixa|99969.06|
|   Caixa|99933.09|
|    Itau|99999.54|
|    Itau|99951.02|
|  Nubank|99935.45|
|  Nubank|99914.35|
|      XP|99961.28|
|      XP|99934.01|
+--------+--------+



In [30]:
df_row_number = spark.sql("""
    select
        destinatario.banco,
        valor,
        row_number() over (partition by destinatario.banco order by valor desc) as row_number
    from transacoes_pix
""")

In [31]:
df_row_number.show()

+-----+--------+----------+
|banco|   valor|row_number|
+-----+--------+----------+
|  BTG|99946.78|         1|
|  BTG| 99913.9|         2|
|  BTG|99873.58|         3|
|  BTG|99865.12|         4|
|  BTG|99840.68|         5|
|  BTG|99832.08|         6|
|  BTG| 99829.9|         7|
|  BTG|99814.23|         8|
|  BTG|99813.42|         9|
|  BTG|99785.91|        10|
|  BTG|99754.22|        11|
|  BTG|99750.69|        12|
|  BTG|99724.27|        13|
|  BTG|99711.66|        14|
|  BTG|99708.06|        15|
|  BTG|99684.07|        16|
|  BTG|99677.36|        17|
|  BTG|99648.38|        18|
|  BTG|99635.23|        19|
|  BTG|99628.33|        20|
+-----+--------+----------+
only showing top 20 rows


In [32]:
from pyspark.sql.functions import col

df_row_number.filter(col('row_number').isin([1,2])).show()

+--------+--------+----------+
|   banco|   valor|row_number|
+--------+--------+----------+
|     BTG|99946.78|         1|
|     BTG| 99913.9|         2|
|Bradesco|99910.87|         1|
|Bradesco|99887.88|         2|
|      C6|99980.03|         1|
|      C6|99964.99|         2|
|   Caixa|99969.06|         1|
|   Caixa|99933.09|         2|
|    Itau|99999.54|         1|
|    Itau|99951.02|         2|
|  Nubank|99935.45|         1|
|  Nubank|99914.35|         2|
|      XP|99961.28|         1|
|      XP|99934.01|         2|
+--------+--------+----------+



# SparkML

In [33]:
# Instalar a última versão do PySpark
!pip install pyspark #==3.3.1

# Instalar o NGROK
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip


# Iniciar a sessão spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4050')
      .appName("SparkSQL")
      .getOrCreate()
)

# Autenticar a sessão do SparkUI com NGROK
!./ngrok authtoken
get_ipython().system_raw('./ngrok http 4050 &')
!sleep 10
!curl -s http://localhost:4040/api/tunnels | grep -Po 'public_url":"(?=https)\K[^"]*'

NAME:
   authtoken - save authtoken to configuration file

USAGE:
   ngrok authtoken [command options] [arguments...]

DESCRIPTION:
   The authtoken command modifies your configuration file to include
   the specified authtoken. By default, this configuration file is located
   at $HOME/.ngrok2/ngrok.yml

   The ngrok.com service requires that you sign up for an account to use
   many advanced service features. In order to associate your client with
   an account, it must pass a secret token to the ngrok.com service when it
   starts up. Instead of passing this authtoken on every invocation, you may
   use this command to save it into your configuration file so that your
   client always authenticates you properly.

EXAMPLE:
    ngrok authtoken BDZIXnhJt2HNWLXyQ5PM_qCaBq0W2sNFcCa0rfTZd

OPTIONS:
   --config 		save in this config file, default: ~/.ngrok2/ngrok.yml
   --log "false"	path to log file, 'stdout', 'stderr' or 'false'
   --log-format "term"	log record format: 'term', 'logfmt',

In [34]:
#from google.colab import drive
#drive.mount('/content/drive')

In [35]:
from pyspark.sql.types import *

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType())
])

schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('chave_pix', StringType()),
    StructField('categoria', StringType()),
    StructField('transaction_date', StringType()),
    StructField('fraude', IntegerType())
])

caminho_json = '/content/pix_transactions.json'

df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

In [36]:
df.printSchema()

df.show()

root
 |-- id_transacao: integer (nullable = true)
 |-- valor: double (nullable = true)
 |-- remetente: struct (nullable = true)
 |    |-- nome: string (nullable = true)
 |    |-- banco: string (nullable = true)
 |    |-- tipo: string (nullable = true)
 |-- destinatario: struct (nullable = true)
 |    |-- nome: string (nullable = true)
 |    |-- banco: string (nullable = true)
 |    |-- tipo: string (nullable = true)
 |-- chave_pix: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- fraude: integer (nullable = true)

+------------+--------+--------------------+--------------------+---------+---------+----------------+------+
|id_transacao|   valor|           remetente|        destinatario|chave_pix|categoria|transaction_date|fraude|
+------------+--------+--------------------+--------------------+---------+---------+----------------+------+
|        1000|    7.05|{Jonathan Gonsalv...|{Gabriel Cunha, I...|      cpf|     

In [37]:
from pyspark.sql.functions import *

df_flatten = df.withColumns({
    'destinatario_nome': col('destinatario').getField('nome'),
    'destinatario_banco': col('destinatario').getField('banco'),
    'destinatario_tipo': col('destinatario').getField('tipo'),
}).drop('remetente', 'destinatario')

In [38]:
df_flatten.printSchema()

root
 |-- id_transacao: integer (nullable = true)
 |-- valor: double (nullable = true)
 |-- chave_pix: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- fraude: integer (nullable = true)
 |-- destinatario_nome: string (nullable = true)
 |-- destinatario_banco: string (nullable = true)
 |-- destinatario_tipo: string (nullable = true)



In [39]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCols=[
        'destinatario_nome',
        'destinatario_banco',
        'destinatario_tipo',
        'categoria',
        'chave_pix'
    ],
    outputCols=[
        'destinatario_nome_index',
        'destinatario_banco_index',
        'destinatario_tipo_index',
        'categoria_index',
        'chave_pix_index'
    ],
    handleInvalid='keep'
)

df_index = indexer.fit(df_flatten).transform(df_flatten)

In [40]:
df_index.printSchema()

root
 |-- id_transacao: integer (nullable = true)
 |-- valor: double (nullable = true)
 |-- chave_pix: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- fraude: integer (nullable = true)
 |-- destinatario_nome: string (nullable = true)
 |-- destinatario_banco: string (nullable = true)
 |-- destinatario_tipo: string (nullable = true)
 |-- destinatario_nome_index: double (nullable = false)
 |-- destinatario_banco_index: double (nullable = false)
 |-- destinatario_tipo_index: double (nullable = false)
 |-- categoria_index: double (nullable = false)
 |-- chave_pix_index: double (nullable = false)



In [41]:
df_index.show()

+------------+--------+---------+---------+----------------+------+--------------------+------------------+-----------------+-----------------------+------------------------+-----------------------+---------------+---------------+
|id_transacao|   valor|chave_pix|categoria|transaction_date|fraude|   destinatario_nome|destinatario_banco|destinatario_tipo|destinatario_nome_index|destinatario_banco_index|destinatario_tipo_index|categoria_index|chave_pix_index|
+------------+--------+---------+---------+----------------+------+--------------------+------------------+-----------------+-----------------------+------------------------+-----------------------+---------------+---------------+
|        1000|    7.05|      cpf|     NULL|      2022-03-19|     0|       Gabriel Cunha|              Itau|               PF|                 8244.0|                     2.0|                    0.0|            0.0|            0.0|
|        1001|   37.28|aleatoria|     NULL|      2021-01-26|     0|         

In [42]:
is_fraud = df_index.filter("fraude == 1") # filter(col('fraude') == 1)
no_fraud = df_index.filter("fraude == 0")

In [43]:
no_fraud = no_fraud.sample(False, 0.01, seed=123)

In [44]:
df_concat = no_fraud.union(is_fraud)
df = df_concat.sort("transaction_date")
df.count()

18320

In [45]:
train, test = df.randomSplit([0.7, 0.3], seed = 123)
print("train =", train.count(), " test =", test.count())

train = 12786  test = 5534


In [46]:
is_fraud = udf(lambda fraud: 1.0 if fraud > 0 else 0.0, DoubleType())
train = train.withColumn("is_fraud", is_fraud(train.fraude))

In [47]:
train.printSchema()

root
 |-- id_transacao: integer (nullable = true)
 |-- valor: double (nullable = true)
 |-- chave_pix: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- fraude: integer (nullable = true)
 |-- destinatario_nome: string (nullable = true)
 |-- destinatario_banco: string (nullable = true)
 |-- destinatario_tipo: string (nullable = true)
 |-- destinatario_nome_index: double (nullable = false)
 |-- destinatario_banco_index: double (nullable = false)
 |-- destinatario_tipo_index: double (nullable = false)
 |-- categoria_index: double (nullable = false)
 |-- chave_pix_index: double (nullable = false)
 |-- is_fraud: double (nullable = true)



In [48]:
train.columns

['id_transacao',
 'valor',
 'chave_pix',
 'categoria',
 'transaction_date',
 'fraude',
 'destinatario_nome',
 'destinatario_banco',
 'destinatario_tipo',
 'destinatario_nome_index',
 'destinatario_banco_index',
 'destinatario_tipo_index',
 'categoria_index',
 'chave_pix_index',
 'is_fraud']

In [49]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols = [x for x in train.columns if x not in ['transaction_date', 'fraude', 'is_fraud', 'destinatario_nome', 'destinatario_banco', 'destinatario_tipo', 'chave_pix', 'categoria']],
    outputCol="features"
)

In [50]:
lr = LogisticRegression().setParams(
    maxIter=100000,
    labelCol = "is_fraud",
    predictionCol="prediction"
)

In [51]:
model = Pipeline(stages=[assembler, lr]).fit(train)

In [52]:
predicted = model.transform(test)

In [53]:
predicted.show()

+------------+--------+---------+---------+----------------+------+--------------------+------------------+-----------------+-----------------------+------------------------+-----------------------+---------------+---------------+--------------------+--------------------+--------------------+----------+
|id_transacao|   valor|chave_pix|categoria|transaction_date|fraude|   destinatario_nome|destinatario_banco|destinatario_tipo|destinatario_nome_index|destinatario_banco_index|destinatario_tipo_index|categoria_index|chave_pix_index|            features|       rawPrediction|         probability|prediction|
+------------+--------+---------+---------+----------------+------+--------------------+------------------+-----------------+-----------------------+------------------------+-----------------------+---------------+---------------+--------------------+--------------------+--------------------+----------+
|        1030|87553.08|    email|     NULL|      2021-02-24|     1|      Gustavo da P

In [54]:
predicted = predicted.withColumn('is_fraud', is_fraud(predicted.fraude))
predicted.crosstab('is_fraud', 'prediction').show()

+-------------------+---+----+
|is_fraud_prediction|0.0| 1.0|
+-------------------+---+----+
|                1.0|  0|5287|
|                0.0|247|   0|
+-------------------+---+----+



In [55]:
df_teste_cols = [
    'id_transacao',
    'valor',
    'transaction_date',
    'destinatario_nome_index',
    'destinatario_banco_index',
    'destinatario_tipo_index',
    'chave_pix_index',
    'categoria_index',
    'fraude'
]

df_teste_data = [
    (999,103.2, "2023-01-01 11:56:41", 328.0, 4.0, 1.0, 3.0, 5.0, 0),
    (998, 500000.0, "2023-01-01 11:56:41", 328.0, 2.0, 3.0, 2.0, 5.0, 1),
    (997, 19999.0, "2023-01-01 11:56:41", 328.0, 1.0, 2.0, 1.0, 5.0, 0),
]

df_teste = spark.createDataFrame(df_teste_data).toDF(*df_teste_cols)

In [56]:
new_prediction = model.transform(df_teste)

In [57]:
new_prediction.show()

+------------+--------+-------------------+-----------------------+------------------------+-----------------------+---------------+---------------+------+--------------------+--------------------+--------------------+----------+
|id_transacao|   valor|   transaction_date|destinatario_nome_index|destinatario_banco_index|destinatario_tipo_index|chave_pix_index|categoria_index|fraude|            features|       rawPrediction|         probability|prediction|
+------------+--------+-------------------+-----------------------+------------------------+-----------------------+---------------+---------------+------+--------------------+--------------------+--------------------+----------+
|         999|   103.2|2023-01-01 11:56:41|                  328.0|                     4.0|                    1.0|            3.0|            5.0|     0|[999.0,103.2,328....|[488.159503346224...|           [1.0,0.0]|       0.0|
|         998|500000.0|2023-01-01 11:56:41|                  328.0|             

#Data Quality

In [1]:
# Instalar a vesão 3.0.3 do PySpark
!pip install pyspark==3.5.0

# Instalar o NGROK
!wget -qnc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -n -q ngrok-stable-linux-amd64.zip

# Autenticar a sessão do SparkUI com NGROK
!./ngrok authtoken
get_ipython().system_raw('./ngrok http 4050 &')
!sleep 10
!curl -s http://localhost:4040/api/tunnels | grep -Po 'public_url":"(?=https)\K[^" ]*'

# from google.colab import drive
# drive.mount('/content/drive')

Authtoken saved to configuration file: /root/.ngrok2/ngrok.yml


In [59]:
!pip install pydeequ

In [2]:
import os

# Set the SPARK_VERSION environment variable for pydeequ compatibility
# Choose the highest supported version by pydeequ that is compatible with your PySpark installation.
# Although PySpark is 4.0.2, pydeequ only explicitly lists up to 3.5 in its error message.
# Setting it to 3.5 is generally a safe bet for compatibility.
os.environ["SPARK_VERSION"] = "3.5"

import pydeequ # pydeequ should be imported after SPARK_VERSION is set
from pyspark.sql import SparkSession

# Stop any existing SparkSession to ensure a fresh session with correct configurations
if 'spark' in locals() and spark.sparkContext._jsc is not None:
    spark.stop()

spark = (
    SparkSession.builder
      .config('spark.ui.port', '4050')
      .config("spark.jars.packages", pydeequ.deequ_maven_coord)
      .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
      .appName("SparkSQL")
      .getOrCreate()
)

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, TimestampType
from pyspark.sql.functions import col

schema_remetente_destinatario = StructType([
    StructField('nome', StringType()),
    StructField('banco', StringType()),
    StructField('tipo', StringType())
])

schema_base_pix = StructType([
    StructField('id_transacao', IntegerType()),
    StructField('valor', DoubleType()),
    StructField('remetente', schema_remetente_destinatario),
    StructField('destinatario', schema_remetente_destinatario),
    StructField('chave_pix', StringType()),
    StructField('categoria', StringType()),
    StructField('transaction_date', StringType()),
    StructField('fraude', IntegerType())
])

caminho_json = '/content/case_final.json'

df = spark.read.json(
    caminho_json,
    schema=schema_base_pix,
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

df = df.withColumn(
      'destinatario_nome', col('destinatario').getField('nome')
    ).withColumn(
      'destinatario_banco', col('destinatario').getField('banco')
    ).withColumn(
      'destinatario_tipo', col('destinatario').getField('tipo')
    ).withColumn(
      'remetente_nome', col('remetente').getField('nome')
    ).withColumn(
      'remetente_banco', col('remetente').getField('banco')
    ).withColumn(
      'remetente_tipo', col('remetente').getField('tipo')
).drop('remetente', 'destinatario')

In [4]:
from pydeequ.analyzers import AnalysisRunner, AnalyzerContext, ApproxCountDistinct, Completeness, Compliance, Mean, Size


analysisResult = (
    AnalysisRunner(spark).onData(df)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness('id_transacao'))
    .addAnalyzer(Compliance("valor", "valor > 0"))
    .run()
)


In [5]:
analysisResult

JavaObject id=o80

In [6]:
analysisResult_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysisResult)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [7]:
analysisResult_df.show()

+-------+------------+------------+--------+
| entity|    instance|        name|   value|
+-------+------------+------------+--------+
|Dataset|           *|        Size|100000.0|
| Column|id_transacao|Completeness|     1.0|
| Column|       valor|  Compliance| 0.99972|
+-------+------------+------------+--------+



In [8]:
from pydeequ.suggestions import ConstraintSuggestionRunner, DEFAULT

suggestionResult = ConstraintSuggestionRunner(spark).onData(df).addConstraintRule(DEFAULT()).run()

In [9]:
for sugg in suggestionResult['constraint_suggestions']:
  print(f"Sugestao de Constraint: \'{sugg['column_name']}\': {sugg['description']}")
  print(f"PySpark Code: {sugg['code_for_constraint']}\n")

Sugestao de Constraint: 'destinatario_nome': 'destinatario_nome' is not null
PySpark Code: .isComplete("destinatario_nome")

Sugestao de Constraint: 'remetente_nome': 'remetente_nome' has value range 'Jonathan Gonsalves'
PySpark Code: .isContainedIn("remetente_nome", ["Jonathan Gonsalves"])

Sugestao de Constraint: 'remetente_nome': 'remetente_nome' is not null
PySpark Code: .isComplete("remetente_nome")

Sugestao de Constraint: 'id_transacao': 'id_transacao' is not null
PySpark Code: .isComplete("id_transacao")

Sugestao de Constraint: 'id_transacao': 'id_transacao' has no negative values
PySpark Code: .isNonNegative("id_transacao")

Sugestao de Constraint: 'id_transacao': 'id_transacao' is unique
PySpark Code: .isUnique("id_transacao")

Sugestao de Constraint: 'remetente_banco': 'remetente_banco' has value range 'BTG'
PySpark Code: .isContainedIn("remetente_banco", ["BTG"])

Sugestao de Constraint: 'remetente_banco': 'remetente_banco' is not null
PySpark Code: .isComplete("remetente_

In [10]:
from pydeequ.checks import Check, CheckLevel, ConstrainableDataTypes
from pydeequ.verification import VerificationResult, VerificationSuite

check = Check(spark, CheckLevel.Warning, "Review Check")
error = Check(spark, CheckLevel.Error, "Error")

In [12]:
checkResult = (
    VerificationSuite(spark)
      .onData(df)
      .addCheck(
        check.hasDataType("id_transacao",ConstrainableDataTypes.Integral)
        .isNonNegative("id_transacao")
        .isComplete("id_transacao")
        .isUnique('id_transacao')
      )
  .run()
)

In [13]:
checkResult_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
checkResult_df.show(truncate=False)

+------------+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|check       |check_level|check_status|constraint                                                                                                                                          |constraint_status|constraint_message|
+------------+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|Review Check|Warning    |Success     |AnalysisBasedConstraint(DataType(id_transacao,None),<function1>,Some(<function1>),None)                                                             |Success          |                  |
|Review Check|Warning    |Success     |ComplianceConstraint(Compliance(id_transacao is non-negat

/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [14]:
checkResult = (
    VerificationSuite(spark)
      .onData(df)
      .addCheck(
        error
          .isContainedIn("remetente_tipo", ["CNPJ"])
      )
  .run()
)

Python Callback server started!


In [15]:
checkResult_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
checkResult_df.show(truncate=False)

+-----+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+----------------------------------------------------+
|check|check_level|check_status|constraint                                                                                                                                                |constraint_status|constraint_message                                  |
+-----+-----------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+----------------------------------------------------+
|Error|Error      |Error       |ComplianceConstraint(Compliance(remetente_tipo contained in CNPJ,`remetente_tipo` IS NULL OR `remetente_tipo` IN ('CNPJ'),None,List(remetente_tipo),None))|Failure          |Value: 0.0 does no